In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# เทคนิคการลดและกดข้อผิดพลาด

> **Note:** เวอร์ชัน beta ของโมเดลการรันแบบใหม่พร้อมให้ใช้งานแล้ว โมเดลการรันแบบ directed ให้ความยืดหยุ่นมากขึ้นในการปรับแต่งเวิร์กโฟลว์การลดข้อผิดพลาด ดูรายละเอียดเพิ่มเติมได้ที่คู่มือ [Directed execution model](/guides/directed-execution-model)


<details>
<summary><b>เวอร์ชันของแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ข้อกำหนดต่อไปนี้
แนะนำให้ใช้เวอร์ชันนี้หรือใหม่กว่า

```
qiskit-ibm-runtime~=0.43.1
```
</details>
เทคนิคการลดข้อผิดพลาด (error mitigation) และการกดข้อผิดพลาด (error suppression) ใช้เพื่อปรับปรุงคุณภาพผลลัพธ์เมื่อขยายงานไปสู่ workload ขนาดใหญ่ขึ้น หน้านี้ให้คำอธิบายระดับสูงเกี่ยวกับเทคนิคการกดข้อผิดพลาดและการลดข้อผิดพลาดที่มีให้ใช้งานผ่าน Qiskit Runtime

เซลล์ต่อไปนี้นำเข้า primitive ของ Estimator และสร้าง backend ที่จะใช้ในการเริ่มต้น Estimator ในเซลล์โค้ดถัดไป

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Dynamical decoupling

Circuit ควอนตัมรันบนฮาร์ดแวร์ IBM&reg; ในรูปแบบลำดับของพัลส์ไมโครเวฟที่ต้องถูกจัดตารางและรันในช่วงเวลาที่แม่นยำ
น่าเสียดายที่ปฏิสัมพันธ์ที่ไม่ต้องการระหว่าง Qubit อาจทำให้เกิดข้อผิดพลาดแบบ coherent บน Qubit ที่กำลังนิ่งอยู่ Dynamical decoupling ทำงานโดยการแทรกลำดับพัลส์บน Qubit ที่นิ่งเพื่อยกเลิกผลของข้อผิดพลาดเหล่านี้โดยประมาณ ลำดับพัลส์แต่ละชุดที่แทรกเข้าไปเทียบเท่ากับการดำเนินการ identity แต่การมีพัลส์จริงทางกายภาพมีผลในการกดข้อผิดพลาด
มีตัวเลือกลำดับพัลส์มากมาย และการเลือกว่าลำดับใดดีกว่าสำหรับกรณีเฉพาะแต่ละกรณียังคงเป็น[พื้นที่การวิจัยที่ยังคงดำเนินต่อ](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027)

โปรดทราบว่า dynamical decoupling มีประโยชน์หลักสำหรับ Circuit ที่มีช่องว่างซึ่ง Qubit บางตัวนิ่งอยู่โดยไม่มีการดำเนินการใดกระทำต่อพวกมัน หากการดำเนินการใน Circuit ถูกอัดแน่นมาก จนทุก Qubit ยุ่งอยู่ตลอดเวลา การเพิ่ม dynamical decoupling pulses อาจไม่ช่วยปรับปรุงประสิทธิภาพ ในความเป็นจริง อาจทำให้ประสิทธิภาพแย่ลงเนื่องจากความไม่สมบูรณ์ของพัลส์เอง

แผนภาพด้านล่างแสดง dynamical decoupling ด้วยลำดับพัลส์ XX Circuit นามธรรมทางซ้ายถูกแมปลงบนตารางเวลาพัลส์ไมโครเวฟที่ด้านบนขวา ด้านล่างขวาแสดงตารางเวลาเดียวกัน แต่มีลำดับ X pulses สองชุดถูกแทรกในช่วงที่ Qubit แรกนิ่ง

![ภาพแสดง dynamical decoupling](../docs/images/guides/error-mitigation-explanation/dd.avif)

สามารถเปิดใช้ Dynamical decoupling ได้โดยตั้ง `enable` เป็น `True` ใน[ตัวเลือก dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options) ตัวเลือก `sequence_type` สามารถใช้เลือกจากลำดับพัลส์หลายประเภท ประเภทลำดับเริ่มต้นคือ `"XX"`

เซลล์โค้ดต่อไปนี้แสดงวิธีเปิดใช้ dynamical decoupling สำหรับ Estimator และเลือกลำดับ dynamical decoupling

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Pauli twirling

Twirling หรือที่เรียกว่า [randomized compiling](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325) เป็นเทคนิคที่ใช้กันอย่างแพร่หลายในการแปลง noise channel ที่ไม่ได้กำหนดเป็น noise channel ที่มีโครงสร้างเฉพาะเจาะจงมากขึ้น

Pauli twirling เป็น twirling ชนิดพิเศษที่ใช้การดำเนินการ Pauli มีผลในการแปลง quantum channel ใดๆ ให้เป็น Pauli channel เมื่อทำแบบเดี่ยวๆ สามารถลด coherent noise ได้เพราะ coherent noise มีแนวโน้มสะสมแบบกำลังสองตามจำนวนการดำเนินการ ในขณะที่ Pauli noise สะสมแบบเชิงเส้น Pauli twirling มักใช้ร่วมกับเทคนิคการลดข้อผิดพลาดอื่นที่ทำงานได้ดีกว่ากับ Pauli noise กว่า noise ทั่วไป

Pauli twirling ถูกนำไปใช้โดยการประกบ Gate ที่เลือกด้วย single-qubit Pauli gates ที่เลือกแบบสุ่ม ในลักษณะที่ผลในอุดมคติของ Gate ยังคงเหมือนเดิม ผลคือ Circuit เดียวถูกแทนที่ด้วย ensemble ของ Circuit สุ่มทั้งหมดที่มีผลในอุดมคติเหมือนกัน เมื่อทำการ sampling Circuit จะมีการดึงตัวอย่างจาก instance สุ่มหลายตัว แทนที่จะเป็นแค่ตัวเดียว

![ภาพแสดง Pauli twirling](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

เนื่องจากข้อผิดพลาดส่วนใหญ่ในฮาร์ดแวร์ควอนตัมปัจจุบันมาจาก two-qubit gates เทคนิคนี้จึงมักถูกใช้กับ (native) two-qubit gates เท่านั้น แผนภาพต่อไปนี้แสดง Pauli twirls บางส่วนสำหรับ Gate CNOT และ ECR ทุก Circuit ในแต่ละแถวมีผลในอุดมคติเหมือนกัน

![ภาพแสดง gate twirls](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

สามารถเปิดใช้ Pauli twirling ได้โดยตั้ง `enable_gates` เป็น `True` ใน[ตัวเลือก twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) ตัวเลือกที่น่าสังเกตอื่นๆ ได้แก่:

- `num_randomizations`: จำนวน instance ของ Circuit ที่จะดึงจาก ensemble ของ twirled circuits
- `shots_per_randomization`: จำนวน shots ที่จะ sample จาก Circuit instance แต่ละตัว

เซลล์โค้ดต่อไปนี้แสดงวิธีเปิดใช้ Pauli twirling และตั้งค่าตัวเลือกเหล่านี้สำหรับ estimator ไม่จำเป็นต้องตั้งค่าตัวเลือกเหล่านี้อย่างชัดเจน

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Twirled readout error extinction (TREX)

[Twirled readout error extinction (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) ลดผลของข้อผิดพลาดในการวัดสำหรับการประมาณค่าความคาดหวังของ Pauli observable
มันอิงจากแนวคิดของการวัดแบบ twirled ซึ่งทำได้โดยการแทนที่ measurement gates แบบสุ่มด้วยลำดับของ (1) Pauli X gate, (2) การวัด และ (3) การพลิกบิตแบบคลาสสิก เช่นเดียวกับ gate twirling มาตรฐาน ลำดับนี้เทียบเท่ากับการวัดธรรมดาเมื่อไม่มี noise ดังที่แสดงในแผนภาพต่อไปนี้:

![ภาพแสดง measurement twirling](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

เมื่อมี readout error การทำ measurement twirling มีผลในการทำให้ readout-error transfer matrix เป็นแบบ diagonal ทำให้ง่ายต่อการหาผกผัน การประมาณ readout-error transfer matrix ต้องรัน calibration circuits เพิ่มเติม ซึ่งทำให้มี overhead เล็กน้อย

สามารถเปิดใช้ TREX ได้โดยตั้ง `measure_mitigation` เป็น `True` ใน[ตัวเลือก Qiskit Runtime resilience](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) สำหรับ Estimator ตัวเลือกสำหรับ measurement noise learning อธิบายไว้[ที่นี่](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options) เช่นเดียวกับ gate twirling สามารถตั้งจำนวน circuit randomizations และจำนวน shots ต่อ randomization ได้

เซลล์โค้ดต่อไปนี้แสดงวิธีเปิดใช้ TREX และตั้งค่าตัวเลือกเหล่านี้สำหรับ estimator ไม่จำเป็นต้องตั้งค่าตัวเลือกเหล่านี้อย่างชัดเจน

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Zero-noise extrapolation (ZNE)
Zero-noise extrapolation (ZNE) เป็นเทคนิคสำหรับลดข้อผิดพลาดในการประมาณค่าความคาดหวังของ observable แม้จะปรับปรุงผลลัพธ์บ่อยครั้ง แต่ไม่รับประกันว่าจะให้ผลลัพธ์ที่ไม่มีอคติ

ZNE ประกอบด้วยสองขั้นตอน:

1. _การขยาย noise_: Circuit ควอนตัมดั้งเดิมถูกรันหลายครั้งในอัตรา noise ที่แตกต่างกัน
2. _การ Extrapolation_: ผลลัพธ์ในอุดมคติถูกประมาณโดยการ extrapolate ผลค่าความคาดหวังที่มี noise ไปยัง zero-noise limit

ทั้งการขยาย noise และขั้นตอน extrapolation สามารถนำไปใช้ได้หลายวิธี Qiskit Runtime นำ noise amplification ไปใช้โดย "digital gate folding" ซึ่งหมายความว่า two-qubit gates ถูกแทนที่ด้วยลำดับที่เทียบเท่าของ Gate และ inverse ของมัน ตัวอย่างเช่น การแทนที่ unitary $U$ ด้วย $U U^\dagger U$ จะให้ noise amplification factor เท่ากับ 3 สำหรับ extrapolation สามารถเลือกจากรูปแบบฟังก์ชันหลายแบบ รวมถึง linear fit หรือ exponential fit
ภาพด้านล่างแสดง digital gate folding ทางซ้าย และขั้นตอน extrapolation ทางขวา

![ภาพแสดง ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

สามารถเปิดใช้ ZNE ได้โดยตั้ง `zne_mitigation` เป็น `True` ใน[ตัวเลือก Qiskit Runtime resilience](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) สำหรับ Estimator
ตัวเลือก Qiskit Runtime สำหรับ ZNE อธิบายไว้[ที่นี่](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) ตัวเลือกที่น่าสังเกตได้แก่:

- `noise_factors`: noise factors ที่จะใช้สำหรับการขยาย noise
- `extrapolator`: รูปแบบฟังก์ชันที่จะใช้สำหรับ extrapolation

เซลล์โค้ดต่อไปนี้แสดงวิธีเปิดใช้ ZNE และตั้งค่าตัวเลือกเหล่านี้สำหรับ estimator ไม่จำเป็นต้องตั้งค่าตัวเลือกเหล่านี้อย่างชัดเจน

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Probabilistic Error Amplification (PEA)
หนึ่งในความท้าทายหลักของ ZNE คือการขยาย noise ที่ส่งผลต่อ Circuit เป้าหมายอย่างแม่นยำ Gate folding ให้วิธีง่ายในการขยายนี้ แต่อาจไม่แม่นยำและอาจนำไปสู่ผลลัพธ์ที่ไม่ถูกต้อง ดูบทความ ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197) และโดยเฉพาะหน้า 4 ของข้อมูลเสริมสำหรับรายละเอียด Probabilistic error amplification ให้แนวทางที่แม่นยำกว่าในการขยาย error ผ่าน noise learning

PEA เป็นเทคนิคที่ซับซ้อนกว่าซึ่งทำการทดลองเบื้องต้นเพื่อสร้างใหม่ noise แล้วใช้ข้อมูลนี้เพื่อทำการขยายที่แม่นยำ มันเริ่มด้วยการเรียนรู้ twirled noise model ของแต่ละเลเยอร์ของ entangling gates ใน Circuit ก่อนที่จะรัน (ดู [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) สำหรับตัวเลือก learning ที่เกี่ยวข้อง) หลังจากระยะการเรียนรู้ Circuit จะถูกรันที่แต่ละ noise factor โดยทุกเลเยอร์ entangling ของ Circuit ถูกขยายโดยการฉีด single-qubit noise แบบ probabilistic ที่สัดส่วนกับ learned noise model ที่สอดคล้องกัน ดูบทความ ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) สำหรับรายละเอียดเพิ่มเติม

PEA ประกอบด้วยสามขั้นตอน:
1. _Learning_: เรียนรู้ twirled noise model ของแต่ละเลเยอร์ของ entangling gates ใน Circuit
1. _การขยาย noise_: Circuit ควอนตัมดั้งเดิมถูกรันหลายครั้งที่ noise factors ที่แตกต่างกัน
2. _Extrapolation_: ผลลัพธ์ในอุดมคติถูกประมาณโดยการ extrapolate ผลค่าความคาดหวังที่มี noise ไปยัง zero-noise limit

สำหรับการทดลองในระดับ utility PEA มักเป็นตัวเลือกที่ดีที่สุด

เนื่องจาก PEA เป็นเทคนิค ZNE noise amplification จึงต้องเปิดใช้ ZNE ด้วยโดยตั้ง `resilience.zne_mitigation = True` ตัวเลือก [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) อื่นๆ สามารถใช้เพิ่มเติมเพื่อตั้ง extrapolators ระดับการขยาย และอื่นๆ PEA ต้องใช้ noise model ซึ่งสร้างขึ้นโดยอัตโนมัติเมื่อใช้ primitives

snippet ต่อไปนี้ให้ตัวอย่างที่ PEA ถูกใช้เพื่อลดผลลัพธ์ของ Estimator job:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Probabilistic error cancellation (PEC)
Probabilistic error cancellation (PEC) เป็นเทคนิคสำหรับลดข้อผิดพลาดในการประมาณค่าความคาดหวังของ observable ต่างจาก ZNE มันส่งคืนการประมาณที่ไม่มีอคติของค่าความคาดหวัง อย่างไรก็ตาม โดยทั่วไปมี overhead ที่มากกว่า

ใน PEC ผลของ Circuit เป้าหมายในอุดมคติถูกแสดงเป็น linear combination ของ Circuit ที่มี noise ซึ่งสามารถนำไปใช้งานได้จริง:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

ผลลัพธ์ของ Circuit ในอุดมคติจึงสามารถจำลองซ้ำได้โดยการรัน Circuit instance ที่มี noise ต่างกันที่ดึงมาจาก ensemble สุ่มที่กำหนดโดย linear combination หากสัมประสิทธิ์ $\eta_i$ ก่อเป็น probability distribution สามารถใช้โดยตรงเป็นความน่าจะเป็นของ ensemble ในทางปฏิบัติ สัมประสิทธิ์บางตัวมีค่าลบ จึงก่อเป็น quasi-probability distribution แทน ยังสามารถใช้กำหนด ensemble สุ่มได้ แต่มี sampling overhead ที่เกี่ยวข้องกับค่าลบของ quasi-probability distribution ซึ่งมีลักษณะเฉพาะด้วยปริมาณ

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

sampling overhead เป็น multiplicative factor ของจำนวน shots ที่ต้องการในการประมาณค่าความคาดหวังถึงความแม่นยำที่กำหนด เมื่อเทียบกับจำนวน shots ที่ต้องการจาก Circuit ในอุดมคติ มันขยายแบบกำลังสองกับ $\gamma$ ซึ่งขยายแบบ exponential กับความลึกของ Circuit

สามารถเปิดใช้ PEC ได้โดยตั้ง `pec_mitigation` เป็น `True` ใน[ตัวเลือก Qiskit Runtime resilience](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) สำหรับ Estimator
ตัวเลือก Qiskit Runtime สำหรับ PEC อธิบายไว้[ที่นี่](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options) สามารถตั้งขีดจำกัดของ sampling overhead ได้โดยใช้ตัวเลือก `max_overhead` โปรดทราบว่าการจำกัด sampling overhead อาจทำให้ความแม่นยำของผลลัพธ์เกินค่าที่ร้องขอ ค่าเริ่มต้นของ `max_overhead` คือ 100

เซลล์โค้ดต่อไปนี้แสดงวิธีเปิดใช้ PEC และตั้งค่าตัวเลือก `max_overhead` สำหรับ estimator

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## ขั้นตอนถัดไป
- ดู[บทเรียน](/tutorials/combine-error-mitigation-techniques)เกี่ยวกับการรวมตัวเลือกการลดข้อผิดพลาดกับ Estimator primitive
- [กำหนดค่าการลดข้อผิดพลาด](configure-error-mitigation)
- [กำหนดค่าการกดข้อผิดพลาด](configure-error-suppression)
- สำรวจ[ตัวเลือก](runtime-options-overview)อื่นๆ สำหรับ Qiskit Runtime primitives
- ตัดสินใจว่าจะใช้[execution mode](execution-modes) ใดในการรัน job ของคุณ